In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import metrics
from constant import EXPS_PATH, PMDATA_PATH, LIFESNAPS_PATH
from scipy.stats import tukey_hsd

In [2]:
def read_data(dataset, fold, flag):
    if dataset == 'lifesnaps':
        data_path = os.path.join(LIFESNAPS_PATH, 'processed', 'top', f'split_{fold}', f'{flag}.parquet')
    elif dataset == 'pmdata':
        data_path = os.path.join(PMDATA_PATH, 'processed', f'split_{fold}', f'{flag}.parquet')
    
    data = pd.read_parquet(data_path)
    return data

In [3]:
def eval_lifesnaps():
    aucs = []
    for fold in range(5):
        print(f'Fold {fold}')
        ls_train = read_data('lifesnaps', fold, 'train')
        ls_test = read_data('lifesnaps', fold, 'test')
        ls_train = ls_train[ls_train['stress_top_y'] != -1]
        ls_test = ls_test[ls_test['stress_top_y'] != -1]
        train_mean = ls_train.groupby('id')['stress_top_y'].mean()
        ls_test = ls_test[['id', 'stress_top_y']].reset_index(drop=True)
        ls_test['stress_top_y_pred'] = None
        for idx in range(len(ls_test)):
            # use the mean of the training data
            id = ls_test.loc[idx, 'id']
            if id not in train_mean:
                print(f'id {id} not in train data')
                continue
            pred = train_mean[id]
            ls_test.loc[idx, 'stress_top_y_pred'] = pred
        ls_test = ls_test.dropna()
        y_true = ls_test['stress_top_y']
        y_pred = ls_test['stress_top_y_pred']
        auc_score = metrics.roc_auc_score(y_true, y_pred)
        aucs.append(auc_score)
    
    mu, sigma = np.mean(aucs), np.std(aucs)
    print(f'${mu*100:.2f} \pm {sigma*100:.2f}$')
    return aucs

eval_lifesnaps()

Fold 0
id 621e375367b776a24021e950 not in train data
id 621e375367b776a24021e950 not in train data
Fold 1
id 621e332267b776a24092a584 not in train data
Fold 2
id 621e34ec67b776a240d60873 not in train data
Fold 3
id 621e32d067b776a2405b7d54 not in train data
id 621e333567b776a240a0c217 not in train data
Fold 4
id 621e2f9167b776a240011ccb not in train data
$80.37 \pm 8.64$


[0.7595238095238095,
 0.8103448275862069,
 0.6648706896551724,
 0.9080459770114943,
 0.8755555555555556]

In [4]:
def eval_pmdata():
    aucs = []
    for fold in range(5):
        pm_train = read_data('pmdata', fold, 'train')
        pm_test = read_data('pmdata', fold, 'test')
        pm_train = pm_train[pm_train['stress_label'] != -1]
        pm_test = pm_test[pm_test['stress_label'] != -1]
        train_mean = pm_train.groupby('participant_id')['stress_label'].mean()
        pm_test = pm_test[['participant_id', 'stress_label']].reset_index(drop=True)
        pm_test['stress_label_pred'] = None
        for idx in range(len(pm_test)):
            # use the mean of the training data
            id = pm_test.loc[idx, 'participant_id']
            if id not in train_mean:
                print(f'id {id} not in train data')
                continue
            pred = train_mean[id]
            pm_test.loc[idx, 'stress_label_pred'] = pred
        pm_test = pm_test.dropna()
        y_true = pm_test['stress_label']
        y_pred = pm_test['stress_label_pred']
        auc_score = metrics.roc_auc_score(y_true, y_pred)
        aucs.append(auc_score)
    
    mu, sigma = np.mean(aucs), np.std(aucs)
    print(f'${mu*100:.2f} \pm {sigma*100:.2f}$')
    return aucs

eval_pmdata()

id p12 not in train data
$84.08 \pm 2.12$


[0.8349751689326712,
 0.8616432700247729,
 0.8044738500315061,
 0.8620703933747412,
 0.8409675443968156]

In [6]:
# read results from ablation studies
def get_exp_path(dataset, exp, chnl):
    aucs = []
    for fold in range(5):
        for seed in range(4):
            exp_path = os.path.join(EXPS_PATH, dataset, exp, f'fold_{fold}', f'seed_{seed}', f'chnl_{chnl}')
            # read results
            zero_shot_results = pd.read_csv(os.path.join(exp_path, 'results.csv'))
            zero_shot_max = zero_shot_results['val_auc'].max()
            linear_prob_results = pd.read_csv(os.path.join(exp_path, 'results_lp.csv'))
            linear_prob_max = linear_prob_results['val_auc'].max()
            fine_tune_results = pd.read_csv(os.path.join(exp_path, 'results_ft.csv'))
            fine_tune_max = fine_tune_results['val_auc'].max()
            auc_max = max(zero_shot_max, linear_prob_max, fine_tune_max)
            aucs.append(auc_max)
    mu, sigma = np.mean(aucs), np.std(aucs)
    print(f'{dataset} {exp} {chnl}')
    print(f'${mu*100:.2f} \pm {sigma*100:.2f}$')
    return aucs

a = get_exp_path('lifesnaps', 'ablation', chnl=0)
b = get_exp_path('lifesnaps', 'ablation', chnl=1)
c = get_exp_path('lifesnaps', 'ablation', chnl=2)
d = get_exp_path('lifesnaps', 'ablation', chnl=3)
e = get_exp_path('lifesnaps', 'ablation', chnl=4)

stats_result = tukey_hsd(a, b, c, d, e)
print(stats_result)

lifesnaps ablation 0
$73.63 \pm 3.38$
lifesnaps ablation 1
$67.84 \pm 4.76$
lifesnaps ablation 2
$69.77 \pm 3.68$
lifesnaps ablation 3
$73.75 \pm 4.72$
lifesnaps ablation 4
$65.52 \pm 5.31$
Tukey's HSD Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)      0.058     0.001     0.018     0.098
 (0 - 2)      0.039     0.063    -0.001     0.079
 (0 - 3)     -0.001     1.000    -0.041     0.039
 (0 - 4)      0.081     0.000     0.041     0.121
 (1 - 0)     -0.058     0.001    -0.098    -0.018
 (1 - 2)     -0.019     0.666    -0.059     0.021
 (1 - 3)     -0.059     0.001    -0.099    -0.019
 (1 - 4)      0.023     0.494    -0.017     0.063
 (2 - 0)     -0.039     0.063    -0.079     0.001
 (2 - 1)      0.019     0.666    -0.021     0.059
 (2 - 3)     -0.040     0.052    -0.080     0.000
 (2 - 4)      0.042     0.032     0.002     0.082
 (3 - 0)      0.001     1.000    -0.039     0.041
 (3 - 1)      0.059     0.001     0.019   

In [7]:
a = get_exp_path('pmdata', 'ablation', chnl=0)
b = get_exp_path('pmdata', 'ablation', chnl=1)
c = get_exp_path('pmdata', 'ablation', chnl=2)
d = get_exp_path('pmdata', 'ablation', chnl=3)

stats_result = tukey_hsd(a, b, c, d)
print(stats_result)

pmdata ablation 0
$81.35 \pm 3.10$
pmdata ablation 1
$81.61 \pm 2.47$
pmdata ablation 2
$81.89 \pm 2.93$
pmdata ablation 3
$81.71 \pm 3.10$
Tukey's HSD Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)     -0.003     0.993    -0.027     0.022
 (0 - 2)     -0.005     0.939    -0.030     0.019
 (0 - 3)     -0.004     0.981    -0.028     0.021
 (1 - 0)      0.003     0.993    -0.022     0.027
 (1 - 2)     -0.003     0.990    -0.028     0.022
 (1 - 3)     -0.001     0.999    -0.026     0.024
 (2 - 0)      0.005     0.939    -0.019     0.030
 (2 - 1)      0.003     0.990    -0.022     0.028
 (2 - 3)      0.002     0.997    -0.023     0.027
 (3 - 0)      0.004     0.981    -0.021     0.028
 (3 - 1)      0.001     0.999    -0.024     0.026
 (3 - 2)     -0.002     0.997    -0.027     0.023

